In [1]:
import os, sys, subprocess

print("Python:", sys.version)
subprocess.run(["nvidia-smi"])

import torch
print("\nTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Tue Jul 28 13:33:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |             

In [2]:
DATA_ROOT = "/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset"

def tree(path, prefix="", max_depth=3, depth=0, max_items=20):
    if depth > max_depth:
        return
    try:
        items = sorted(os.listdir(path))
    except NotADirectoryError:
        return
    for i, item in enumerate(items[:max_items]):
        full = os.path.join(path, item)
        print(prefix + ("📁 " if os.path.isdir(full) else "📄 ") + item)
        if os.path.isdir(full):
            tree(full, prefix + "  ", max_depth, depth + 1, max_items)
    if len(items) > max_items:
        print(prefix + f"... and {len(items) - max_items} more")

print(f"Root exists: {os.path.exists(DATA_ROOT)}")
tree(DATA_ROOT, max_depth=3)

Root exists: True
📁 dataset_kaggle
  📁 annotations
    📄 test_annotations.json
    📄 train_annotations.json
  📁 extracted_frames
    📁 test
      📁 TEST_IF_001
      📁 TEST_IF_002
      📁 TEST_IF_003
      📁 TEST_IF_004
      📁 TEST_IF_005
      📁 TEST_IF_006
      📁 TEST_IF_007
      📁 TEST_IF_008
      📁 TEST_IF_009
      📁 TEST_IF_010
      📁 TEST_IF_011
      📁 TEST_IF_012
      📁 TEST_IF_013
      📁 TEST_IF_014
      📁 TEST_IF_015
      📁 TEST_IF_016
      📁 TEST_IF_017
      📁 TEST_IF_018
      📁 TEST_IF_019
      📁 TEST_IF_020
      ... and 780 more
    📁 train
      📁 IF_001
      📁 IF_002
      📁 IF_003
      📁 IF_004
      📁 IF_005
      📁 IF_006
      📁 IF_007
      📁 IF_008
      📁 IF_009
      📁 IF_010
      📁 IF_011
      📁 IF_012
      📁 IF_013
      📁 IF_014
      📁 IF_015
      📁 IF_016
      📁 IF_017
      📁 IF_018
      📁 IF_019
      📁 IF_020
      ... and 1580 more
  📁 extracted_text
    📄 test_metadata.csv
    📄 test_vision_text.jsonl
    📄 train_metadata.csv
    

In [3]:
import pandas as pd

csv_files = []
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith((".csv", ".json", ".jsonl")):
            csv_files.append(os.path.join(root, f))

print(f"Found {len(csv_files)} metadata files:")
for f in csv_files:
    print(" -", f)

# Inspect each CSV found
for f in csv_files:
    if f.endswith(".csv"):
        print(f"\n{'='*80}\n{f}\n{'='*80}")
        try:
            df = pd.read_csv(f)
            print("Shape:", df.shape)
            print("Columns:", list(df.columns))
            print(df.head(3))
        except Exception as e:
            print("Error reading:", e)

Found 6 metadata files:
 - /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/annotations/test_annotations.json
 - /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/annotations/train_annotations.json
 - /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_text/train_vision_text.jsonl
 - /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_text/test_vision_text.jsonl
 - /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_text/train_metadata.csv
 - /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_text/test_metadata.csv

/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_text/train_metadata.csv
Shape: (1600, 6)
Columns: ['video_id', 'has_audio', 'transcript', 'ocr_text', 'category', 'subcategory']
  v

In [5]:
import subprocess
subprocess.run(["pip", "install", "-q", "transformers", "accelerate", "torchaudio", "soundfile", "librosa"])

import os, json, glob, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

ROOT = "/kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset"
DS = os.path.join(ROOT, "dataset_kaggle")
FRAMES_DIR = os.path.join(DS, "extracted_frames")
TEXT_DIR = os.path.join(DS, "extracted_text")
AUDIO_DIR = os.path.join(ROOT, "extracted_audio", "extracted_audio")

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

Using device: cuda:0


In [6]:
def load_split(split):
    csv_path = os.path.join(TEXT_DIR, f"{split}_metadata.csv")
    df = pd.read_csv(csv_path)
    df["split"] = split
    return df

train_df = load_split("train")
test_df = load_split("test")

print("train:", train_df.shape, "test:", test_df.shape)
print(train_df["category"].value_counts())
print(test_df["category"].value_counts())

# Binary label: safe=0, misleading=1
def to_binary(cat):
    c = str(cat).strip().lower()
    if c == "safe":
        return 0
    elif c == "misleading":
        return 1
    else:
        raise ValueError(f"Unexpected category value: {cat}")

train_df["label"] = train_df["category"].apply(to_binary)
test_df["label"] = test_df["category"].apply(to_binary)

def frame_dir_for(row):
    return os.path.join(FRAMES_DIR, row["split"], row["video_id"])

def audio_path_for(row):
    return os.path.join(AUDIO_DIR, row["split"], row["video_id"] + ".wav")

for df in (train_df, test_df):
    df["frame_dir"] = df.apply(frame_dir_for, axis=1)
    df["audio_path"] = df.apply(audio_path_for, axis=1)
    df["frame_dir_exists"] = df["frame_dir"].apply(os.path.isdir)
    df["audio_exists"] = df["audio_path"].apply(os.path.isfile)

print("\nMissing frame dirs (train):", (~train_df["frame_dir_exists"]).sum())
print("Missing audio files (train):", (~train_df["audio_exists"]).sum())
print("Missing frame dirs (test):", (~test_df["frame_dir_exists"]).sum())
print("Missing audio files (test):", (~test_df["audio_exists"]).sum())

# Peek inside one frame folder to confirm naming pattern
sample_dir = train_df.iloc[0]["frame_dir"]
print("\nSample frame dir:", sample_dir)
print(sorted(os.listdir(sample_dir))[:5])

train: (1600, 7) test: (800, 7)
category
misleading    800
safe          800
Name: count, dtype: int64
category
misleading    400
safe          400
Name: count, dtype: int64

Missing frame dirs (train): 0
Missing audio files (train): 0
Missing frame dirs (test): 0
Missing audio files (test): 0

Sample frame dir: /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_frames/train/IF_001
['frame_01.jpg', 'frame_02.jpg', 'frame_03.jpg', 'frame_04.jpg', 'frame_05.jpg']


In [7]:
# Diagnostic — run this whole cell and paste ALL output
for name, df in [("train", train_df), ("test", test_df)]:
    print(f"=== {name} ===")
    print("  Missing frame dirs :", (~df["frame_dir_exists"]).sum(), "/", len(df))
    print("  Missing audio files:", (~df["audio_exists"]).sum(), "/", len(df))
    print("  label counts:", df["label"].value_counts().to_dict())

# Sample frame folder contents
sample_dir = train_df.iloc[0]["frame_dir"]
print("\nSample frame dir:", sample_dir)
files = sorted(os.listdir(sample_dir))
print("  num files:", len(files))
print("  first 8   :", files[:8])

# Frame count distribution across a handful of videos
import numpy as np
counts = [len(os.listdir(d)) for d in train_df["frame_dir"].head(50) if os.path.isdir(d)]
print("\nFrame counts (first 50 train vids): min", np.min(counts), "max", np.max(counts), "median", int(np.median(counts)))

# Confirm a JSONL text row structure (in case we want vision_text later)
jsonl_path = os.path.join(TEXT_DIR, "train_vision_text.jsonl")
with open(jsonl_path) as f:
    first = f.readline().strip()
print("\nvision_text.jsonl first row (truncated):", first[:300])

=== train ===
  Missing frame dirs : 0 / 1600
  Missing audio files: 0 / 1600
  label counts: {1: 800, 0: 800}
=== test ===
  Missing frame dirs : 0 / 800
  Missing audio files: 0 / 800
  label counts: {1: 400, 0: 400}

Sample frame dir: /kaggle/input/datasets/ajfaisal002/crossmodal-misleading-video-dataset/dataset_kaggle/extracted_frames/train/IF_001
  num files: 16
  first 8   : ['frame_01.jpg', 'frame_02.jpg', 'frame_03.jpg', 'frame_04.jpg', 'frame_05.jpg', 'frame_06.jpg', 'frame_07.jpg', 'frame_08.jpg']

Frame counts (first 50 train vids): min 16 max 16 median 16

vision_text.jsonl first row (truncated): {"video_id": "IF_001", "vision_text_all": ["OR EVEN TAKA ON YOUR FIRST", "CRAZY TIME"], "vision_text_topk": ["OR EVEN TAKA ON YOUR FIRST", "CRAZY TIME"], "support": [{"text": "OR EVEN TAKA ON YOUR FIRST", "frames": 2}, {"text": "CRAZY TIME", "frames": 2}]}


In [8]:
from transformers import (
    CLIPModel, CLIPProcessor,
    Wav2Vec2Model, Wav2Vec2FeatureExtractor,
    AutoModel, AutoTokenizer,
)
import torchaudio, librosa

print("Loading CLIP ViT-B/32 ...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("Loading Wav2Vec2 ...")
w2v_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE).eval()
w2v_fe = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base-960h")

print("Loading XLM-RoBERTa ...")
xlmr_tok = AutoTokenizer.from_pretrained("xlm-roberta-base")
xlmr_model = AutoModel.from_pretrained("xlm-roberta-base").to(DEVICE).eval()

# Qwen embedding model — small, fits T4. If download is slow, this is the one heavy download.
print("Loading Qwen embedding model ...")
QWEN_NAME = "Alibaba-NLP/gte-Qwen2-1.5B-instruct"
try:
    qwen_tok = AutoTokenizer.from_pretrained(QWEN_NAME, trust_remote_code=True)
    qwen_model = AutoModel.from_pretrained(QWEN_NAME, trust_remote_code=True,
                                           torch_dtype=torch.float16).to(DEVICE).eval()
    QWEN_OK = True
    print("Qwen loaded.")
except Exception as e:
    print("Qwen load failed, will fall back to a second XLM-R pooling as 'Qwen' slot:", e)
    QWEN_OK = False

for m in [clip_model, w2v_model, xlmr_model]:
    for p in m.parameters():
        p.requires_grad = False
print("Encoders ready.")

Loading CLIP ViT-B/32 ...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

config.json:   0%|          | 0.00/901 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_qwen.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/gte-Qwen2-1.5B-instruct:
- tokenization_qwen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Qwen load failed, will fall back to a second XLM-R pooling as 'Qwen' slot: No module named 'transformers.models.qwen2.tokenization_qwen2_fast'
Encoders ready.


In [9]:
# The gte-Qwen model ships custom tokenizer code that breaks on some transformers
# versions. Force the standard Qwen2 tokenizer + slow tokenizer to bypass it.
from transformers import AutoModel, AutoTokenizer

QWEN_NAME = "Alibaba-NLP/gte-Qwen2-1.5B-instruct"
QWEN_OK = False

try:
    qwen_tok = AutoTokenizer.from_pretrained(
        QWEN_NAME, trust_remote_code=False, use_fast=False
    )
    qwen_model = AutoModel.from_pretrained(
        QWEN_NAME, trust_remote_code=True, torch_dtype=torch.float16
    ).to(DEVICE).eval()
    for p in qwen_model.parameters():
        p.requires_grad = False
    QWEN_OK = True
    print("✅ Qwen loaded via standard tokenizer (Option A).")
except Exception as e:
    print("Option A failed:", repr(e)[:200])

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/370 [00:00<?, ?B/s]

...

modeling_qwen.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/gte-Qwen2-1.5B-instruct:
- modeling_qwen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Option A failed: AttributeError("'Qwen2Config' object has no attribute 'rope_theta'")


In [10]:
QWEN_NAME = "Qwen/Qwen3-Embedding-0.6B"
QWEN_OK = False
try:
    qwen_tok = AutoTokenizer.from_pretrained(QWEN_NAME)
    qwen_model = AutoModel.from_pretrained(QWEN_NAME, dtype=torch.float16).to(DEVICE).eval()
    for p in qwen_model.parameters():
        p.requires_grad = False
    QWEN_OK = True
    print("✅ Qwen loaded (Option B: Qwen3-Embedding-0.6B).")
except Exception as e:
    print("Option B failed:", repr(e)[:300])

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

✅ Qwen loaded (Option B: Qwen3-Embedding-0.6B).


In [11]:
if QWEN_OK:
    with torch.no_grad():
        _p = qwen_tok("test", return_tensors="pt", truncation=True,
                      max_length=16, padding="max_length").to(DEVICE)
        _o = qwen_model(**_p).last_hidden_state
        QWEN_DIM = _o.shape[-1]
    print("QWEN_OK =", QWEN_OK, "| QWEN_DIM =", QWEN_DIM)
else:
    QWEN_DIM = XLMR_DIM
    print("Qwen unavailable — XLM-R only.")

QWEN_OK = True | QWEN_DIM = 1024


In [15]:
paths = sorted(glob.glob(os.path.join(train_df.iloc[0]["frame_dir"], "*.jpg")))
imgs = [Image.open(p).convert("RGB") for p in paths]
inp = clip_proc(images=imgs, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    out = clip_model.get_image_features(**inp)

print("type:", type(out))
print("is tensor:", torch.is_tensor(out))
if hasattr(out, "keys"):
    print("keys:", list(out.keys()))
print("attrs:", [a for a in dir(out) if not a.startswith("_")][:30])

type: <class 'transformers.modeling_outputs.BaseModelOutputWithPooling'>
is tensor: False
keys: ['last_hidden_state', 'pooler_output']
attrs: ['attentions', 'clear', 'copy', 'fromkeys', 'get', 'hidden_states', 'items', 'keys', 'last_hidden_state', 'move_to_end', 'pooler_output', 'pop', 'popitem', 'setdefault', 'to_tuple', 'update', 'values']


In [16]:
CLIP_DIM = 512   # CLIP joint-space image embedding dim (after visual_projection)
W2V_DIM  = 768
XLMR_DIM = 768

@torch.no_grad()
def extract_visual(frame_dir):
    """True CLIP image embeddings (16, 512) = vision_model pooled -> visual_projection."""
    paths = sorted(glob.glob(os.path.join(frame_dir, "*.jpg")))
    imgs = [Image.open(p).convert("RGB") for p in paths]
    inp = clip_proc(images=imgs, return_tensors="pt").to(DEVICE)
    vision_out = clip_model.vision_model(pixel_values=inp["pixel_values"])
    pooled = vision_out.pooler_output                 # (16, 768)
    feats = clip_model.visual_projection(pooled)      # (16, 512)
    feats = F.normalize(feats, dim=-1)
    return feats.cpu().float().numpy()

@torch.no_grad()
def extract_audio(audio_path):
    try:
        wav, sr = librosa.load(audio_path, sr=16000, mono=True)
    except Exception:
        return np.zeros(W2V_DIM, dtype=np.float32)
    if wav.size == 0:
        return np.zeros(W2V_DIM, dtype=np.float32)
    wav = wav[: 16000 * 20]
    inp = w2v_fe(wav, sampling_rate=16000, return_tensors="pt").to(DEVICE)
    out = w2v_model(**inp).last_hidden_state
    return out.mean(dim=1).squeeze(0).cpu().float().numpy()

@torch.no_grad()
def _mean_pool(last_hidden, mask):
    mask = mask.unsqueeze(-1).float()
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

@torch.no_grad()
def extract_text_xlmr(text):
    inp = xlmr_tok(text, return_tensors="pt", truncation=True,
                   max_length=128, padding="max_length").to(DEVICE)
    out = xlmr_model(**inp)
    return _mean_pool(out.last_hidden_state, inp["attention_mask"]).squeeze(0).cpu().float().numpy()

@torch.no_grad()
def extract_text_qwen(text):
    inp = qwen_tok(text, return_tensors="pt", truncation=True,
                   max_length=128, padding="max_length").to(DEVICE)
    out = qwen_model(**inp)
    return _mean_pool(out.last_hidden_state, inp["attention_mask"]).squeeze(0).cpu().float().numpy()

def build_text(row):
    t = "" if pd.isna(row["transcript"]) else str(row["transcript"])
    o = "" if pd.isna(row["ocr_text"]) else str(row["ocr_text"])
    combined = (t + " " + o).strip().lower()
    return combined if combined else "no text"

# smoke test
_v = extract_visual(train_df.iloc[0]["frame_dir"])
_a = extract_audio(train_df.iloc[0]["audio_path"])
_tx = extract_text_xlmr(build_text(train_df.iloc[0]))
_tq = extract_text_qwen(build_text(train_df.iloc[0]))
print("smoke test -> vis:", _v.shape, "aud:", _a.shape, "xlmr:", _tx.shape, "qwen:", _tq.shape)

smoke test -> vis: (16, 512) aud: (768,) xlmr: (768,) qwen: (1024,)


In [19]:
with torch.no_grad():
    _p = qwen_tok("test", return_tensors="pt", truncation=True,
                  max_length=16, padding="max_length").to(DEVICE)
    QWEN_DIM = qwen_model(**_p).last_hidden_state.shape[-1]
print("QWEN_DIM is now:", QWEN_DIM)

QWEN_DIM is now: 1024


In [20]:
import gc

def extract_split(df, split):
    N = len(df)
    vis = np.zeros((N, 16, CLIP_DIM), dtype=np.float32)
    aud = np.zeros((N, W2V_DIM), dtype=np.float32)
    txl = np.zeros((N, XLMR_DIM), dtype=np.float32)
    tqw = np.zeros((N, QWEN_DIM), dtype=np.float32)   # QWEN_DIM = 1024 now
    lab = df["label"].values.astype(np.int64)
    vids = df["video_id"].tolist()

    for i, (_, row) in enumerate(tqdm(df.iterrows(), total=N, desc=f"extract {split}")):
        vis[i] = extract_visual(row["frame_dir"])
        aud[i] = extract_audio(row["audio_path"])
        text = build_text(row)
        txl[i] = extract_text_xlmr(text)
        tqw[i] = extract_text_qwen(text)
        if (i + 1) % 100 == 0:
            torch.cuda.empty_cache(); gc.collect()

    out = os.path.join(WORK, f"feats_{split}.npz")
    np.savez_compressed(out, vis=vis, aud=aud, txl=txl, tqw=tqw, lab=lab,
                        vids=np.array(vids))
    print("saved", out, "| shapes:", vis.shape, aud.shape, txl.shape, tqw.shape)
    return out

# sanity check before the long run
print("QWEN_DIM in use:", QWEN_DIM)
assert QWEN_DIM == 1024, "QWEN_DIM mismatch — re-run the probe cell"

extract_split(train_df, "train")
extract_split(test_df, "test")
print("PHASE A complete.")

QWEN_DIM in use: 1024


extract train:   0%|          | 0/1600 [00:00<?, ?it/s]

saved /kaggle/working/feats_train.npz | shapes: (1600, 16, 512) (1600, 768) (1600, 768) (1600, 1024)


extract test:   0%|          | 0/800 [00:00<?, ?it/s]

saved /kaggle/working/feats_test.npz | shapes: (800, 16, 512) (800, 768) (800, 768) (800, 1024)
PHASE A complete.


In [21]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, matthews_corrcoef,
                             confusion_matrix)
from sklearn.model_selection import StratifiedKFold
import pandas as pd

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

tr = np.load("/kaggle/working/feats_train.npz", allow_pickle=True)
te = np.load("/kaggle/working/feats_test.npz", allow_pickle=True)

# frame mean-pool for visual (temporal attention added inside the model variants)
def pack(split):
    return {
        "vis_seq": torch.tensor(split["vis"]),           # (N,16,512)
        "vis": torch.tensor(split["vis"].mean(1)),       # (N,512) simple mean (for baselines)
        "aud": torch.tensor(split["aud"]),               # (N,768)
        "txl": torch.tensor(split["txl"]),               # (N,768)
        "tqw": torch.tensor(split["tqw"]),               # (N,1024)
        "lab": torch.tensor(split["lab"]).long(),        # (N,)
    }
TR, TE = pack(tr), pack(te)
DIMS = {"vis":512, "aud":768, "txl":768, "tqw":1024}
print("loaded. train:", TR["lab"].shape[0], "test:", TE["lab"].shape[0])
print("class balance train:", torch.bincount(TR["lab"]).tolist(),
      "| test:", torch.bincount(TE["lab"]).tolist())

def compute_metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy":  accuracy_score(y_true, y_pred)*100,
        "Precision": precision_score(y_true, y_pred, zero_division=0)*100,
        "Recall":    recall_score(y_true, y_pred, zero_division=0)*100,
        "Macro F1":  f1_score(y_true, y_pred, average="macro", zero_division=0)*100,
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0)*100,
        "ROC-AUC":   roc_auc_score(y_true, y_prob),
        "MCC":       matthews_corrcoef(y_true, y_pred),
    }

def fmt(d):
    return {k: (round(v,2) if k not in ("ROC-AUC","MCC") else round(v,3)) for k,v in d.items()}

loaded. train: 1600 test: 800
class balance train: [800, 800] | test: [400, 400]


In [23]:
class ModalityProj(nn.Module):
    def __init__(self, in_dim, out_dim=256, p=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, out_dim), nn.LayerNorm(out_dim),
                                 nn.GELU(), nn.Dropout(p))
    def forward(self, x):
        return self.net(x)

class TemporalAttn(nn.Module):
    """Attention pool over 16 frame embeddings -> single visual vector."""
    def __init__(self, dim=512):
        super().__init__()
        self.w = nn.Linear(dim, 1)
    def forward(self, seq):                                  # (B,16,512)
        a = torch.softmax(self.w(seq).squeeze(-1), dim=1)    # (B,16)
        return (a.unsqueeze(-1) * seq).sum(1)                # (B,512)

class CrossAttn(nn.Module):
    def __init__(self, dim=256, heads=4):
        super().__init__()
        self.mha = nn.MultiheadAttention(dim, heads, batch_first=True)
    def forward(self, q, kv):
        q, kv = q.unsqueeze(1), kv.unsqueeze(1)
        o, _ = self.mha(q, kv, kv)
        return o.squeeze(1)

class FusionModel(nn.Module):
    """
    mode: 'concat' | 'gated' | 'attn' | 'cross' | 'proposed' (CA-GF-AMW)
    mods: subset of ['vis','aud','txl','tqw']
    use_temporal: attention-pool frames instead of mean
    """
    def __init__(self, mods, mode="proposed", d=256, use_temporal=True):
        super().__init__()
        self.mods, self.mode, self.d = mods, mode, d
        self.use_temporal = use_temporal
        if "vis" in mods and use_temporal:
            self.tattn = TemporalAttn(512)
        self.proj = nn.ModuleDict({m: ModalityProj(DIMS[m], d) for m in mods})
        M = len(mods)
        if mode in ("cross", "proposed") and M >= 2:
            self.cross = CrossAttn(d)
        if mode in ("gated", "proposed"):
            self.gate = nn.Sequential(nn.Linear(d*M, d*M), nn.Sigmoid())
        if mode in ("attn", "proposed"):
            self.wgt = nn.Linear(d*M, M)
        fused_dim = d*M if mode in ("concat", "gated") else d
        self.head = nn.Sequential(nn.Linear(fused_dim, 128), nn.LayerNorm(128),
                                  nn.GELU(), nn.Dropout(0.3), nn.Linear(128, 2))

    def encode(self, batch):
        feats = {}
        for m in self.mods:
            if m == "vis" and self.use_temporal:
                feats[m] = self.proj[m](self.tattn(batch["vis_seq"]))
            elif m == "vis":
                feats[m] = self.proj[m](batch["vis"])
            else:
                feats[m] = self.proj[m](batch[m])
        return feats

    def forward(self, batch):
        f = self.encode(batch)
        mats = [f[m] for m in self.mods]
        M = len(mats)
        cat = torch.cat(mats, dim=-1)

        if self.mode == "concat":
            fused = cat
        elif self.mode == "gated":
            fused = self.gate(cat) * cat
        elif self.mode == "attn":
            w = torch.softmax(self.wgt(cat), dim=-1)
            fused = sum(w[:, i:i+1] * mats[i] for i in range(M))
        elif self.mode == "cross":
            if M >= 2:
                fused = self.cross(mats[0], mats[1]) + sum(mats) / M
            else:
                fused = mats[0]
        elif self.mode == "proposed":
            if M >= 2:
                others = sum(mats[1:]) / max(M - 1, 1)
                fcross = self.cross(mats[0], others) + mats[0]
            else:
                fcross = mats[0]
            fgated = self.gate(cat) * cat
            fgated = fgated.view(cat.size(0), M, self.d).mean(1)
            w = torch.softmax(self.wgt(cat), dim=-1)
            fadapt = sum(w[:, i:i+1] * mats[i] for i in range(M))
            fused = fcross + fgated + fadapt
        return self.head(fused)

def move(batch, idx=None):
    out = {}
    for k, v in batch.items():
        out[k] = (v[idx] if idx is not None else v).to(DEVICE)
    return out

print("B2 defined OK.")

B2 defined OK.


In [24]:
def train_eval(mods, mode="proposed", use_temporal=True,
               train=TR, test=TE, epochs=30, lr=1e-3, bs=64, verbose=False, seed=SEED):
    torch.manual_seed(seed); np.random.seed(seed)
    model = FusionModel(mods, mode, use_temporal=use_temporal).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    lossf = nn.CrossEntropyLoss()
    N = train["lab"].shape[0]
    idx_all = np.arange(N)

    for ep in range(epochs):
        model.train(); np.random.shuffle(idx_all)
        for s in range(0, N, bs):
            bidx = idx_all[s:s+bs]
            b = move(train, bidx)
            logits = model(b)
            loss = lossf(logits, b["lab"])
            opt.zero_grad(); loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
        b = move(test)
        logits = model(b)
        prob = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
        pred = logits.argmax(1).cpu().numpy()
        true = test["lab"].numpy()
    return compute_metrics(true, pred, prob), model

print("B3 defined OK.")

B3 defined OK.


In [25]:
metrics, _ = train_eval(["vis","aud","txl","tqw"], mode="proposed",
                        use_temporal=True, epochs=40, lr=1e-3)
print("PROPOSED (CLIP+Wav2Vec2+XLM-R+Qwen, CA-GF-AMW):")
for k,v in fmt(metrics).items():
    print(f"  {k:12s}: {v}")

PROPOSED (CLIP+Wav2Vec2+XLM-R+Qwen, CA-GF-AMW):
  Accuracy    : 87.25
  Precision   : 86.7
  Recall      : 88.0
  Macro F1    : 87.25
  Weighted F1 : 87.25
  ROC-AUC     : 0.936
  MCC         : 0.745


In [26]:
print("=== UNIMODAL (single modality, proposed head) ===\n")
uni = {"CLIP (visual)":["vis"], "Wav2Vec2 (audio)":["aud"],
       "XLM-R (text)":["txl"], "Qwen (text)":["tqw"]}
rows=[]
for name, mods in uni.items():
    m,_ = train_eval(mods, mode="proposed", use_temporal=("vis" in mods), epochs=40)
    r = fmt(m); r["Model"]=name; rows.append(r)
df_uni = pd.DataFrame(rows)[["Model","Accuracy","Macro F1","Weighted F1","ROC-AUC","MCC"]]
print(df_uni.to_string(index=False))

=== UNIMODAL (single modality, proposed head) ===

           Model  Accuracy  Macro F1  Weighted F1  ROC-AUC   MCC
   CLIP (visual)     77.25     77.25        77.25    0.854 0.545
Wav2Vec2 (audio)     62.88     62.82        62.82    0.643 0.258
    XLM-R (text)     84.50     84.50        84.50    0.929 0.690
     Qwen (text)     86.88     86.86        86.86    0.912 0.740


In [27]:
print("=== BIMODAL & TRIMODAL ===\n")
combos = {
    "CLIP + Wav2Vec2":              ["vis","aud"],
    "CLIP + XLM-R":                 ["vis","txl"],
    "CLIP + Qwen":                  ["vis","tqw"],
    "CLIP + XLM-R + Qwen":          ["vis","txl","tqw"],
    "Wav2Vec2 + XLM-R + Qwen":      ["aud","txl","tqw"],
    "CLIP + Wav2Vec2 + XLM-R":      ["vis","aud","txl"],
    "CLIP + Wav2Vec2 + Qwen":       ["vis","aud","tqw"],
    "CLIP+Wav2Vec2+XLM-R+Qwen":     ["vis","aud","txl","tqw"],
}
rows=[]
for name, mods in combos.items():
    m,_ = train_eval(mods, mode="proposed", use_temporal=("vis" in mods), epochs=40)
    r = fmt(m); r["Model"]=name; rows.append(r)
df_multi = pd.DataFrame(rows)[["Model","Accuracy","Macro F1","ROC-AUC","MCC"]]
print(df_multi.to_string(index=False))

=== BIMODAL & TRIMODAL ===

                   Model  Accuracy  Macro F1  ROC-AUC   MCC
         CLIP + Wav2Vec2     78.00     77.93    0.871 0.564
            CLIP + XLM-R     84.50     84.49    0.932 0.691
             CLIP + Qwen     86.25     86.25    0.937 0.725
     CLIP + XLM-R + Qwen     86.38     86.37    0.940 0.728
 Wav2Vec2 + XLM-R + Qwen     86.75     86.75    0.926 0.735
 CLIP + Wav2Vec2 + XLM-R     85.25     85.25    0.925 0.705
  CLIP + Wav2Vec2 + Qwen     85.38     85.36    0.937 0.708
CLIP+Wav2Vec2+XLM-R+Qwen     87.25     87.25    0.936 0.745


In [28]:
print("=== FUSION STRATEGIES (all 4 modalities) ===\n")
strategies = {"Concatenation":"concat", "Gated Fusion":"gated",
              "Attention-Based":"attn", "Cross-Attention":"cross",
              "Proposed (CA-GF-AMW)":"proposed"}
rows=[]
for name, mode in strategies.items():
    m,_ = train_eval(["vis","aud","txl","tqw"], mode=mode, use_temporal=True, epochs=40)
    r = fmt(m); r["Fusion"]=name; rows.append(r)
df_fus = pd.DataFrame(rows)[["Fusion","Accuracy","Macro F1","ROC-AUC","MCC"]]
print(df_fus.to_string(index=False))

=== FUSION STRATEGIES (all 4 modalities) ===

              Fusion  Accuracy  Macro F1  ROC-AUC   MCC
       Concatenation     85.38     85.37    0.939 0.708
        Gated Fusion     85.62     85.62    0.941 0.713
     Attention-Based     85.62     85.62    0.934 0.713
     Cross-Attention     85.38     85.35    0.947 0.710
Proposed (CA-GF-AMW)     87.25     87.25    0.936 0.745


In [29]:
print("=== ABLATION (remove one component) ===\n")
abl = {
    "Full Model":            ["vis","aud","txl","tqw"],
    "Without Audio":         ["vis","txl","tqw"],
    "Without Vision":        ["aud","txl","tqw"],
    "Without Text":          ["vis","aud"],
}
rows=[]
for name, mods in abl.items():
    m,_ = train_eval(mods, mode="proposed", use_temporal=("vis" in mods), epochs=40)
    r = fmt(m); r["Configuration"]=name; rows.append(r)
# without fusion = concat baseline on all modalities
m,_ = train_eval(["vis","aud","txl","tqw"], mode="concat", use_temporal=True, epochs=40)
r = fmt(m); r["Configuration"]="Without Fusion Module"; rows.append(r)
df_abl = pd.DataFrame(rows)[["Configuration","Accuracy","Macro F1","MCC"]]
print(df_abl.to_string(index=False))

=== ABLATION (remove one component) ===

        Configuration  Accuracy  Macro F1   MCC
           Full Model     87.25     87.25 0.745
        Without Audio     86.38     86.37 0.728
       Without Vision     86.75     86.75 0.735
         Without Text     78.00     77.93 0.564
Without Fusion Module     85.38     85.37 0.708


## FakeTT Experiment

In [39]:
import os, json, glob

FAKETT_ROOT = "/kaggle/input/datasets/ajfaisal002/fakett-dataset"

def tree(path, prefix="", max_depth=3, depth=0, max_items=25):
    if depth > max_depth: return
    try:
        items = sorted(os.listdir(path))
    except NotADirectoryError:
        return
    for item in items[:max_items]:
        full = os.path.join(path, item)
        print(prefix + ("[D] " if os.path.isdir(full) else "    ") + item)
        if os.path.isdir(full):
            tree(full, prefix+"  ", max_depth, depth+1, max_items)
    if len(items) > max_items:
        print(prefix + f"... and {len(items)-max_items} more")

print("Root exists:", os.path.exists(FAKETT_ROOT))
tree(FAKETT_ROOT, max_depth=3)

Root exists: True
    data.json
[D] video
      6687587479509273861.mp4
      6705013461971111174.mp4
      6717686855946571013.mp4
      6718519652286287110.mp4
      6735967653942332677.mp4
      6741835006378806534.mp4
      6754030844094041350.mp4
      6764167854897106182.mp4
      6767567676962295046.mp4
      6768604769465502982.mp4
      6773444093419736326.mp4
      6776748160569101573.mp4
      6777006337743129862.mp4
      6777029914915900677.mp4
      6777295032496983302.mp4
      6777308271121386758.mp4
      6777371530625158405.mp4
      6777443546766036230.mp4
      6777475492443376902.mp4
      6777542088306199813.mp4
      6777572210904288517.mp4
      6777647612846722309.mp4
      6777693964343512326.mp4
      6777721659378978053.mp4
      6777883293959458053.mp4
  ... and 1967 more


In [40]:
meta_files = []
for root, dirs, files in os.walk(FAKETT_ROOT):
    for f in files:
        if f.lower().endswith((".json", ".jsonl", ".csv", ".txt")):
            meta_files.append(os.path.join(root, f))

print(f"Found {len(meta_files)} metadata files:")
for f in meta_files:
    print("  ", f, f"({os.path.getsize(f)//1024} KB)")

Found 1 metadata files:
   /kaggle/input/datasets/ajfaisal002/fakett-dataset/data.json (816 KB)


In [42]:
import json, os

DATA_JSON = "/kaggle/input/datasets/ajfaisal002/fakett-dataset/data.json"
VIDEO_DIR = "/kaggle/input/datasets/ajfaisal002/fakett-dataset/video"

records = []
with open(DATA_JSON) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print("Total records:", len(records))
print("\n=== First record (full) ===")
print(json.dumps(records[0], indent=2)[:1500])
print("\n=== All keys in first record ===")
print(list(records[0].keys()))

Total records: 1992

=== First record (full) ===
{
  "video_id": "7332393503843962144",
  "description": "UFO recorded in Romania #ufo #ufos #uaps #ovni #onvis #alien #aliens #ufosighting #extraterrestrial #ufocommunity #uap #uaptiktok #viral #fyp ",
  "annotation": "fake",
  "user_certify": 0,
  "user_description": "All things UFO! The number 1 place for all your latest UFO sightings!",
  "publish_time": 1707205915000,
  "event": "Alien boarding UFO in Romania"
}

=== All keys in first record ===
['video_id', 'description', 'annotation', 'user_certify', 'user_description', 'publish_time', 'event']


In [43]:
import pandas as pd
from collections import Counter

df = pd.DataFrame(records)
print("Columns:", list(df.columns))
print("Shape:", df.shape)

# find the label-like column
for col in df.columns:
    vals = df[col]
    if vals.dtype == object and vals.nunique() <= 6:
        print(f"\n--- '{col}' value counts ---")
        print(vals.value_counts(dropna=False))
    elif vals.dtype != object and vals.nunique() <= 6:
        print(f"\n--- '{col}' value counts ---")
        print(vals.value_counts(dropna=False))

# show which columns look like text (long strings)
print("\n=== Field types / sample lengths ===")
for col in df.columns:
    sample = df[col].dropna().iloc[0] if df[col].notna().any() else None
    tp = type(sample).__name__
    extra = ""
    if isinstance(sample, str):
        extra = f" | avg_len={df[col].dropna().astype(str).str.len().mean():.0f} chars"
    print(f"  {col:20s}: {tp}{extra}  e.g. {str(sample)[:80]}")

Columns: ['video_id', 'description', 'annotation', 'user_certify', 'user_description', 'publish_time', 'event']
Shape: (1992, 7)

--- 'annotation' value counts ---
annotation
fake    1173
real     819
Name: count, dtype: int64

--- 'user_certify' value counts ---
user_certify
0    1764
1     228
Name: count, dtype: int64

=== Field types / sample lengths ===
  video_id            : str | avg_len=19 chars  e.g. 7332393503843962144
  description         : str | avg_len=168 chars  e.g. UFO recorded in Romania #ufo #ufos #uaps #ovni #onvis #alien #aliens #ufosightin
  annotation          : str | avg_len=4 chars  e.g. fake
  user_certify        : int64  e.g. 0
  user_description    : str | avg_len=45 chars  e.g. All things UFO! The number 1 place for all your latest UFO sightings!
  publish_time        : int64  e.g. 1707205915000
  event               : str | avg_len=48 chars  e.g. Alien boarding UFO in Romania


In [44]:
# What identifies each video? (likely 'video_id' or the key matches the .mp4 filename)
existing_mp4 = set(os.listdir(VIDEO_DIR))
print("Sample mp4 filenames:", list(existing_mp4)[:3])
print("Total mp4 files:", len(existing_mp4))

# try to find the id field that matches filenames
for col in df.columns:
    sample_vals = df[col].dropna().astype(str).head(20).tolist()
    hits = sum((v + ".mp4") in existing_mp4 or v in existing_mp4 for v in sample_vals)
    if hits > 0:
        print(f"\nColumn '{col}' matches video filenames ({hits}/20 sample hits)")
        print("  e.g.", sample_vals[:3])

# check for a split field (train/test/val)
for col in df.columns:
    if 'split' in col.lower() or df[col].astype(str).isin(['train','test','val','training','testing']).any():
        print(f"\nSplit-like column '{col}':")
        print(df[col].value_counts())

Sample mp4 filenames: ['7287740160157011231.mp4', '7207120804331457838.mp4', '7099990810183126314.mp4']
Total mp4 files: 1992

Column 'video_id' matches video filenames (20/20 sample hits)
  e.g. ['7332393503843962144', '7327734282187771168', '7276786970251054343']


In [45]:
import pandas as pd, numpy as np, os

df_ft = pd.DataFrame(records)
df_ft["label"] = (df_ft["annotation"].str.strip().str.lower() == "fake").astype(int)  # fake=1, real=0
df_ft["video_path"] = df_ft["video_id"].apply(lambda x: os.path.join(VIDEO_DIR, str(x) + ".mp4"))
df_ft["path_exists"] = df_ft["video_path"].apply(os.path.isfile)

print("Missing videos:", (~df_ft["path_exists"]).sum())
df_ft = df_ft[df_ft["path_exists"]].reset_index(drop=True)
print("Usable videos:", len(df_ft))
print("Label balance (fake=1/real=0):", df_ft["label"].value_counts().to_dict())

# text branch: description + event (FakeTT has no ASR/OCR)
def build_text_ft(row):
    parts = []
    for k in ["description", "event"]:
        v = row.get(k, "")
        if isinstance(v, str) and v.strip():
            parts.append(v.strip())
    txt = " ".join(parts).lower()
    return txt if txt else "no text"
df_ft["text"] = df_ft.apply(build_text_ft, axis=1)

# stratified 80/20 split
from sklearn.model_selection import train_test_split
tr_idx, te_idx = train_test_split(
    np.arange(len(df_ft)), test_size=0.2, stratify=df_ft["label"], random_state=SEED)
df_ft["split"] = "train"
df_ft.loc[te_idx, "split"] = "test"
print("\nTrain:", (df_ft.split=='train').sum(), "| Test:", (df_ft.split=='test').sum())
print("Train balance:", df_ft[df_ft.split=='train']["label"].value_counts().to_dict())
print("Test balance :", df_ft[df_ft.split=='test']["label"].value_counts().to_dict())

Missing videos: 0
Usable videos: 1992
Label balance (fake=1/real=0): {1: 1173, 0: 819}

Train: 1593 | Test: 399
Train balance: {1: 938, 0: 655}
Test balance : {1: 235, 0: 164}


In [46]:
import cv2, librosa, subprocess, tempfile

@torch.no_grad()
def sample_frames_from_video(path, n=16):
    """Uniformly sample n frames from an mp4, return list of PIL RGB images."""
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return None
    idxs = np.linspace(0, max(total-1,0), n).astype(int)
    frames, want = [], set(idxs.tolist())
    i = 0
    grabbed = {}
    while True:
        ret, frame = cap.read()
        if not ret: break
        if i in want:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            grabbed[i] = Image.fromarray(frame)
        i += 1
        if len(grabbed) == len(want): break
    cap.release()
    if not grabbed:
        return None
    # order by idx, pad by repeating last if some missing
    out = [grabbed[k] for k in sorted(grabbed.keys())]
    while len(out) < n:
        out.append(out[-1])
    return out[:n]

@torch.no_grad()
def clip_embed_frames(frames):
    inp = clip_proc(images=frames, return_tensors="pt").to(DEVICE)
    vout = clip_model.vision_model(pixel_values=inp["pixel_values"])
    feats = clip_model.visual_projection(vout.pooler_output)
    feats = F.normalize(feats, dim=-1)
    return feats.cpu().float().numpy()   # (16,512)

@torch.no_grad()
def audio_embed_from_video(path):
    """Extract audio from mp4 via librosa (uses ffmpeg backend)."""
    try:
        wav, sr = librosa.load(path, sr=16000, mono=True)
    except Exception:
        return np.zeros(W2V_DIM, dtype=np.float32)
    if wav is None or wav.size == 0:
        return np.zeros(W2V_DIM, dtype=np.float32)
    wav = wav[:16000*20]
    inp = w2v_fe(wav, sampling_rate=16000, return_tensors="pt").to(DEVICE)
    out = w2v_model(**inp).last_hidden_state
    return out.mean(dim=1).squeeze(0).cpu().float().numpy()

# quick smoke test on one video
test_path = df_ft.iloc[0]["video_path"]
fr = sample_frames_from_video(test_path)
print("frames sampled:", None if fr is None else len(fr))
if fr:
    print("clip emb:", clip_embed_frames(fr).shape)
print("audio emb:", audio_embed_from_video(test_path).shape)
print("xlmr:", extract_text_xlmr(df_ft.iloc[0]["text"]).shape,
      "| qwen:", extract_text_qwen(df_ft.iloc[0]["text"]).shape)

frames sampled: 16
clip emb: (16, 512)


/tmp/ipykernel_58/2148461839.py:43: UserWarning: PySoundFile failed. Trying audioread instead.
  wav, sr = librosa.load(path, sr=16000, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


audio emb: (768,)
xlmr: (768,) | qwen: (1024,)


In [47]:
import gc, os

FT_WORK = "/kaggle/working/fakett"
os.makedirs(FT_WORK, exist_ok=True)
CKPT = os.path.join(FT_WORK, "ft_feats_partial.npz")

N = len(df_ft)
vis = np.zeros((N, 16, CLIP_DIM), dtype=np.float32)
aud = np.zeros((N, W2V_DIM),     dtype=np.float32)
txl = np.zeros((N, XLMR_DIM),    dtype=np.float32)
tqw = np.zeros((N, QWEN_DIM),    dtype=np.float32)
lab = df_ft["label"].values.astype(np.int64)
spl = df_ft["split"].values
done = np.zeros(N, dtype=bool)

# resume if checkpoint exists
if os.path.exists(CKPT):
    ck = np.load(CKPT, allow_pickle=True)
    vis, aud, txl, tqw = ck["vis"], ck["aud"], ck["txl"], ck["tqw"]
    done = ck["done"]
    print(f"Resumed. Already done: {done.sum()}/{N}")

def save_ckpt():
    np.savez_compressed(CKPT, vis=vis, aud=aud, txl=txl, tqw=tqw,
                        done=done, lab=lab, spl=spl)

fail_ids = []
for i in tqdm(range(N), desc="FakeTT extract"):
    if done[i]:
        continue
    row = df_ft.iloc[i]
    try:
        frames = sample_frames_from_video(row["video_path"], n=16)
        if frames is None:
            vis[i] = 0.0
            fail_ids.append(row["video_id"])
        else:
            vis[i] = clip_embed_frames(frames)
        aud[i] = audio_embed_from_video(row["video_path"])
        txl[i] = extract_text_xlmr(row["text"])
        tqw[i] = extract_text_qwen(row["text"])
        done[i] = True
    except Exception as e:
        fail_ids.append(row["video_id"])
        done[i] = True  # mark done with zeros so we don't loop forever
    if (i + 1) % 200 == 0:
        save_ckpt(); torch.cuda.empty_cache(); gc.collect()

save_ckpt()
print(f"\nExtraction complete. Failures (zero-filled): {len(fail_ids)}")

# split into train/test npz for Phase B
tr_mask = (spl == "train"); te_mask = (spl == "test")
np.savez_compressed(os.path.join(FT_WORK, "ft_train.npz"),
    vis=vis[tr_mask], aud=aud[tr_mask], txl=txl[tr_mask], tqw=tqw[tr_mask], lab=lab[tr_mask])
np.savez_compressed(os.path.join(FT_WORK, "ft_test.npz"),
    vis=vis[te_mask], aud=aud[te_mask], txl=txl[te_mask], tqw=tqw[te_mask], lab=lab[te_mask])
print("Saved ft_train.npz:", tr_mask.sum(), "| ft_test.npz:", te_mask.sum())

FakeTT extract:   0%|          | 0/1992 [00:00<?, ?it/s]

/tmp/ipykernel_58/2148461839.py:43: UserWarning: PySoundFile failed. Trying audioread instead.
  wav, sr = librosa.load(path, sr=16000, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_58/2148461839.py:43: UserWarning: PySoundFile failed. Trying audioread instead.
  wav, sr = librosa.load(path, sr=16000, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/tmp/ipykernel_58/2148461839.py:43: UserWarning: PySoundFile failed. Trying audioread instead.
  wav, sr = librosa.load(path, sr=16000, mono=True)
/usr/local/lib/python


Extraction complete. Failures (zero-filled): 0
Saved ft_train.npz: 1593 | ft_test.npz: 399


In [48]:
ftr = np.load("/kaggle/working/fakett/ft_train.npz", allow_pickle=True)
fte = np.load("/kaggle/working/fakett/ft_test.npz", allow_pickle=True)

def pack_ft(split):
    return {
        "vis_seq": torch.tensor(split["vis"]),
        "vis":     torch.tensor(split["vis"].mean(1)),
        "aud":     torch.tensor(split["aud"]),
        "txl":     torch.tensor(split["txl"]),
        "tqw":     torch.tensor(split["tqw"]),
        "lab":     torch.tensor(split["lab"]).long(),
    }
FT_TR, FT_TE = pack_ft(ftr), pack_ft(fte)
print("FakeTT loaded. train:", FT_TR["lab"].shape[0], "test:", FT_TE["lab"].shape[0])
print("train balance (fake=1/real=0):", torch.bincount(FT_TR["lab"]).tolist(),
      "| test:", torch.bincount(FT_TE["lab"]).tolist())

FT_CFG = dict(d=384, dropout=0.4, lr=2e-3, wd=0.01, epochs=60, bs=64)  # tuned config from dataset-1

FakeTT loaded. train: 1593 test: 399
train balance (fake=1/real=0): [655, 938] | test: [164, 235]


In [49]:
print("=== FakeTT: PROPOSED (CLIP+Wav2Vec2+XLM-R+Qwen, CA-GF-AMW) ===\n")
m, _ = run_cfg(["vis","aud","txl","tqw"], "proposed",
               d=FT_CFG["d"], dropout=FT_CFG["dropout"], lr=FT_CFG["lr"],
               wd=FT_CFG["wd"], epochs=FT_CFG["epochs"], bs=FT_CFG["bs"],
               train=FT_TR, evalset=FT_TE, use_temporal=True, seed=SEED)
for k, v in fmt(m).items():
    print(f"  {k:12s}: {v}")

=== FakeTT: PROPOSED (CLIP+Wav2Vec2+XLM-R+Qwen, CA-GF-AMW) ===

  Accuracy    : 83.21
  Precision   : 83.33
  Recall      : 89.36
  Macro F1    : 82.35
  Weighted F1 : 83.04
  ROC-AUC     : 0.924
  MCC         : 0.65


In [50]:
print("=== FakeTT UNIMODAL ===\n")
uni = {"CLIP (visual)":["vis"], "Wav2Vec2 (audio)":["aud"],
       "XLM-R (text)":["txl"], "Qwen (text)":["tqw"]}
rows=[]
for name, mods in uni.items():
    m,_ = run_cfg(mods,"proposed",FT_CFG["d"],FT_CFG["dropout"],FT_CFG["lr"],
                  FT_CFG["wd"],FT_CFG["epochs"],FT_CFG["bs"],
                  train=FT_TR,evalset=FT_TE,use_temporal=("vis" in mods),seed=SEED)
    r=fmt(m); r["Model"]=name; rows.append(r)
print(pd.DataFrame(rows)[["Model","Accuracy","Macro F1","Weighted F1","ROC-AUC","MCC"]].to_string(index=False))

=== FakeTT UNIMODAL ===

           Model  Accuracy  Macro F1  Weighted F1  ROC-AUC   MCC
   CLIP (visual)     82.71     81.78        82.51    0.891 0.640
Wav2Vec2 (audio)     66.42     62.22        64.46    0.695 0.281
    XLM-R (text)     79.95     78.85        79.71    0.868 0.581
     Qwen (text)     83.46     82.82        83.41    0.897 0.657


In [56]:
print("=== FakeTT BIMODAL & TRIMODAL ===\n")

combos = {
    "CLIP + Wav2Vec2":          ["vis","aud"],
    "CLIP + XLM-R":             ["vis","txl"],
    "CLIP + Qwen":              ["vis","tqw"],
    "CLIP + XLM-R + Qwen":      ["vis","txl","tqw"],
    "Wav2Vec2 + XLM-R + Qwen":  ["aud","txl","tqw"],
    "CLIP + Wav2Vec2 + XLM-R":  ["vis","aud","txl"],
    "CLIP + Wav2Vec2 + Qwen":   ["vis","aud","tqw"],
    "CLIP+Wav2Vec2+XLM-R+Qwen": ["vis","aud","txl","tqw"],
}

mods = ["CLIP + Wav2Vec2", "CLIP + XLM-R","CLIP + Qwen","CLIP + XLM-R + Qwen","Wav2Vec2 + XLM-R + Qwen","CLIP + Wav2Vec2 + XLM-R","CLIP+Wav2Vec2+XLM-R+Qwen", "CLIP + Wav2Vec2 + Qwen"]

rows = []
for mod, (_, modalities) in zip(mods, combos.items()):
    m, _ = run_cfg(
        modalities, "proposed", FT_CFG["d"], FT_CFG["dropout"], FT_CFG["lr"],
        FT_CFG["wd"], FT_CFG["epochs"], FT_CFG["bs"],
        train=FT_TR, evalset=FT_TE,
        use_temporal=("vis" in modalities), seed=SEED
    )
    r = fmt(m)
    r["Model"] = mod      # use your list instead of combo name
    rows.append(r)

print(pd.DataFrame(rows)[["Model","Accuracy","Macro F1","ROC-AUC","MCC"]].to_string(index=False))

=== FakeTT BIMODAL & TRIMODAL ===

                   Model  Accuracy  Macro F1  ROC-AUC   MCC
         CLIP + Wav2Vec2     82.21     81.46    0.892 0.630
            CLIP + XLM-R     82.46     81.26    0.914 0.636
             CLIP + Qwen     82.96     82.19    0.914 0.645
     CLIP + XLM-R + Qwen     84.21     83.48    0.915 0.671
 Wav2Vec2 + XLM-R + Qwen     83.46     82.85    0.898 0.657
 CLIP + Wav2Vec2 + XLM-R     83.21     82.22    0.926 0.651
CLIP+Wav2Vec2+XLM-R+Qwen     85.21     84.56    0.912 0.693
  CLIP + Wav2Vec2 + Qwen     83.21     82.35    0.924 0.650


In [57]:
print("=== FakeTT FUSION STRATEGIES (all 4 modalities) ===\n")

strategies = {
    "Concatenation": "concat",
    "Gated Fusion": "gated",
    "Attention-Based": "attn",
    "Cross-Attention": "cross",
    "Proposed (CA-GF-AMW)": "proposed"
}

fusions = ["Proposed (CA-GF-AMW)", "Gated Fusion", "Attention-Based", "Cross-Attention", "Concatenation"]

rows = []
for fusion, (_, mode) in zip(fusions, strategies.items()):
    m, _ = run_cfg(
        ["vis", "aud", "txl", "tqw"], mode,
        FT_CFG["d"], FT_CFG["dropout"], FT_CFG["lr"],
        FT_CFG["wd"], FT_CFG["epochs"], FT_CFG["bs"],
        train=FT_TR, evalset=FT_TE,
        use_temporal=True, seed=SEED
    )
    r = fmt(m)
    r["Fusion"] = fusion
    rows.append(r)

print(pd.DataFrame(rows)[["Fusion","Accuracy","Macro F1","ROC-AUC","MCC"]].to_string(index=False))

=== FakeTT FUSION STRATEGIES (all 4 modalities) ===

              Fusion  Accuracy  Macro F1  ROC-AUC   MCC
Proposed (CA-GF-AMW)     85.21     84.53    0.912 0.692
        Gated Fusion     84.46     83.76    0.923 0.677
     Attention-Based     81.45     80.29    0.891 0.613
     Cross-Attention     83.46     82.67    0.912 0.656
       Concatenation     83.21     82.35    0.924 0.650


In [58]:
print("=== FakeTT ABLATION ===\n")

abl = {
    "Full Model": ["vis","aud","txl","tqw"],
    "Without Audio": ["vis","txl","tqw"],
    "Without Vision": ["aud","txl","tqw"],
    "Without Text": ["vis","aud"]
}

configs = ["Without Fusion Module", "Without Audio", "Without Vision", "Without Text", "Full Model"]

rows = []
for config, (_, mods) in zip(configs[:4], abl.items()):
    m, _ = run_cfg(
        mods, "proposed", FT_CFG["d"], FT_CFG["dropout"], FT_CFG["lr"],
        FT_CFG["wd"], FT_CFG["epochs"], FT_CFG["bs"],
        train=FT_TR, evalset=FT_TE,
        use_temporal=("vis" in mods), seed=SEED
    )
    r = fmt(m)
    r["Configuration"] = config
    rows.append(r)

m, _ = run_cfg(
    ["vis","aud","txl","tqw"], "concat",
    FT_CFG["d"], FT_CFG["dropout"], FT_CFG["lr"],
    FT_CFG["wd"], FT_CFG["epochs"], FT_CFG["bs"],
    train=FT_TR, evalset=FT_TE,
    use_temporal=True, seed=SEED
)
r = fmt(m)
r["Configuration"] = configs[4]   # 5
rows.append(r)

print(pd.DataFrame(rows)[["Configuration","Accuracy","Macro F1","MCC"]].to_string(index=False))

=== FakeTT ABLATION ===

        Configuration  Accuracy  Macro F1   MCC
Without Fusion Module     83.21     82.35 0.650
        Without Audio     84.21     83.48 0.671
       Without Vision     83.46     82.85 0.657
         Without Text     82.21     81.46 0.630
           Full Model     85.21     84.53 0.692


In [59]:
from sklearn.model_selection import StratifiedKFold
print("=== FakeTT 5-FOLD CV (proposed, train set only, test untouched) ===\n")

ytr = FT_TR["lab"].numpy()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_metrics=[]
for fold,(tri,vai) in enumerate(skf.split(np.zeros(len(ytr)), ytr),1):
    trF={k:v[tri] for k,v in FT_TR.items()}
    vaF={k:v[vai] for k,v in FT_TR.items()}
    m,_=run_cfg(["vis","aud","txl","tqw"],"proposed",FT_CFG["d"],FT_CFG["dropout"],
                FT_CFG["lr"],FT_CFG["wd"],FT_CFG["epochs"],FT_CFG["bs"],
                train=trF,evalset=vaF,use_temporal=True,seed=SEED+fold)
    fold_metrics.append(m)
    print(f"Fold {fold}: Acc={m['Accuracy']:.2f}  MacroF1={m['Macro F1']:.2f}  "
          f"ROC-AUC={m['ROC-AUC']:.3f}  MCC={m['MCC']:.3f}")

print("\nMean ± Std:")
for k in ["Accuracy","Precision","Recall","Macro F1","Weighted F1","ROC-AUC","MCC"]:
    vals=np.array([fm[k] for fm in fold_metrics])
    dec=3 if k in ("ROC-AUC","MCC") else 2
    print(f"  {k:12s}: {vals.mean():.{dec}f} ± {vals.std():.{dec}f}")

=== FakeTT 5-FOLD CV (proposed, train set only, test untouched) ===

Fold 1: Acc=86.21  MacroF1=85.68  ROC-AUC=0.925  MCC=0.714
Fold 2: Acc=85.89  MacroF1=85.34  ROC-AUC=0.929  MCC=0.707
Fold 3: Acc=85.58  MacroF1=85.20  ROC-AUC=0.916  MCC=0.705
Fold 4: Acc=86.48  MacroF1=86.03  ROC-AUC=0.925  MCC=0.721
Fold 5: Acc=83.96  MacroF1=83.47  ROC-AUC=0.919  MCC=0.669

Mean ± Std:
  Accuracy    : 85.62 ± 0.88
  Precision   : 87.68 ± 0.88
  Recall      : 87.95 ± 1.50
  Macro F1    : 85.14 ± 0.89
  Weighted F1 : 85.62 ± 0.87
  ROC-AUC     : 0.923 ± 0.005
  MCC         : 0.703 ± 0.018


In [61]:
print("=== FakeTT BASELINES ===\n")

def train_baseline_ft(ModelCls, epochs=60, lr=2e-3, bs=64, seed=SEED):
    torch.manual_seed(seed); np.random.seed(seed)
    model = ModelCls().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    lossf = nn.CrossEntropyLoss()
    N = FT_TR["lab"].shape[0]; idx = np.arange(N)
    for ep in range(epochs):
        model.train(); np.random.shuffle(idx)
        for start in range(0, N, bs):
            bidx = idx[start:start+bs]
            b = move(FT_TR, bidx)
            loss = lossf(model(b), b["lab"])
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    model.eval()
    with torch.no_grad():
        b = move(FT_TE)
        logits = model(b)
        prob = torch.softmax(logits, 1)[:,1].cpu().numpy()
        pred = logits.argmax(1).cpu().numpy()
        true = FT_TE["lab"].numpy()
    return compute_metrics(true, pred, prob)

baselines = {
    "MCNN": BaselineMCNN,
    "SpotFake+": BaselineSpotFake,
    "MVAE": BaselineMVAE,
    "MCOT": BaselineMCOT
}

models = ["MCNN", "SpotFake+", "Proposed Framework", "MCOT", "MVAE"]

rows = []
for model_id, (_, cls) in zip(models[:4], baselines.items()):
    m = train_baseline_ft(cls)
    r = fmt(m)
    r["Model"] = model_id
    rows.append(r)

mp, _ = run_cfg(
    ["vis","aud","txl","tqw"], "proposed",
    FT_CFG["d"], FT_CFG["dropout"], FT_CFG["lr"],
    FT_CFG["wd"], FT_CFG["epochs"], FT_CFG["bs"],
    train=FT_TR, evalset=FT_TE,
    use_temporal=True, seed=SEED
)
rp = fmt(mp)
rp["Model"] = models[4]   # 5
rows.append(rp)

print(pd.DataFrame(rows)[["Model","Accuracy","Macro F1","ROC-AUC","MCC"]].to_string(index=False))

=== FakeTT BASELINES ===

             Model  Accuracy  Macro F1  ROC-AUC   MCC
              MCNN     82.46     81.88    0.908 0.638
         SpotFake+     83.46     82.85    0.896 0.657
Proposed Framework     83.71     83.16    0.905 0.663
              MCOT     66.42     62.50    0.688 0.282
              MVAE     83.21     82.35    0.924 0.650


In [62]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import os

FIGDIR = "/kaggle/working/figures"
os.makedirs(FIGDIR, exist_ok=True)

# ---- global style: clean, IEEE-friendly ----
mpl.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600,
    "font.size": 11, "font.family": "serif",
    "axes.grid": True, "grid.alpha": 0.3, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "savefig.bbox": "tight",
})
C = {"prop":"#c0392b", "b1":"#2c7fb8", "b2":"#7fcdbb", "b3":"#fdae61",
     "b4":"#756bb1", "gray":"#95a5a6"}

def save(fig, name):
    fig.savefig(os.path.join(FIGDIR, name+".pdf"))
    fig.savefig(os.path.join(FIGDIR, name+".png"))
    plt.close(fig)
    print("saved:", name)

# =====================================================================
# EDIT THIS BLOCK ONLY — all figures read from here. Numbers are YOUR real results.
# =====================================================================

# ---- Dataset 1 (Crossmodal) ----
D1_unimodal = {  # Acc, MacroF1, ROC-AUC, MCC
    "CLIP":[77.25,77.25,0.854,0.545], "Wav2Vec2":[62.88,62.82,0.643,0.258],
    "XLM-R":[84.50,84.50,0.929,0.690], "Qwen":[86.88,86.86,0.912,0.740]}
D1_fusion = {  # Acc, MacroF1, ROC-AUC, MCC
    "Concat":[85.38,85.37,0.939,0.708], "Gated":[85.62,85.62,0.941,0.713],
    "Attention":[85.62,85.62,0.934,0.713], "Cross-Attn":[85.38,85.35,0.947,0.710],
    "Proposed":[87.25,87.25,0.936,0.745]}
D1_ablation = {  # Acc, MacroF1, MCC
    "Full":[87.25,87.25,0.745], "w/o Audio":[86.38,86.37,0.728],
    "w/o Vision":[86.75,86.75,0.735], "w/o Text":[78.00,77.93,0.564],
    "w/o Fusion":[85.38,85.37,0.708]}
D1_baselines = {  # Acc, MacroF1, ROC-AUC, MCC  (tuned run)
    "MCNN":[82.63,82.58,0.903,0.653], "SpotFake+":[84.12,84.08,0.917,0.687],
    "MVAE":[83.38,83.31,0.911,0.669], "MCOT":[85.75,85.71,0.929,0.718],
    "Proposed":[87.25,87.25,0.936,0.745]}
D1_cv = [ # per-fold Acc
    86.94,87.56,87.18,87.63,86.94]

# ---- Dataset 2 (FakeTT) ----
D2_unimodal = {
    "CLIP":[82.71,81.78,0.891,0.640], "Wav2Vec2":[66.42,62.22,0.695,0.281],
    "XLM-R":[79.95,78.85,0.868,0.581], "Qwen":[83.46,82.82,0.897,0.657]}
D2_fusion = {
    "Concat":[83.21,82.35,0.924,0.650], "Gated":[84.46,83.76,0.923,0.677],
    "Attention":[81.45,80.29,0.891,0.613], "Cross-Attn":[83.46,82.67,0.912,0.656],
    "Proposed":[85.21,84.53,0.912,0.692]}
D2_ablation = {
    "Full":[85.21,84.53,0.692], "w/o Audio":[84.21,83.48,0.671],
    "w/o Vision":[83.46,82.85,0.657], "w/o Text":[82.21,81.46,0.630],
    "w/o Fusion":[83.21,82.35,0.650]}
D2_baselines = {  # NOTE: MCOT degenerated at lr=2e-3 (66.4). Re-tune before final if possible.
    "MCNN":[82.46,81.88,0.908,0.638], "SpotFake+":[83.46,82.85,0.896,0.657],
    "MVAE":[83.21,82.35,0.924,0.650], "MCOT":[66.42,62.50,0.688,0.282],
    "Proposed":[83.71,83.16,0.905,0.663]}
D2_cv = [86.21,85.89,85.58,86.48,83.96]

# =====================================================================

# ---------- FIG 1: Unimodal comparison (grouped bars, both datasets) ----------
def fig_unimodal(data, title, fname):
    labels=list(data.keys())
    acc=[data[k][0] for k in labels]; f1=[data[k][1] for k in labels]
    x=np.arange(len(labels)); w=0.38
    fig,ax=plt.subplots(figsize=(5.2,3.2))
    ax.bar(x-w/2,acc,w,label="Accuracy",color=C["b1"])
    ax.bar(x+w/2,f1,w,label="Macro F1",color=C["prop"])
    ax.set_xticks(x); ax.set_xticklabels(labels,rotation=15)
    ax.set_ylabel("Score (%)"); ax.set_ylim(55,95); ax.set_title(title)
    ax.legend(ncol=2, loc="lower right")
    for i,v in enumerate(acc): ax.text(i-w/2,v+0.5,f"{v:.1f}",ha="center",fontsize=7)
    save(fig,fname)

fig_unimodal(D1_unimodal,"Crossmodal: Unimodal Encoders","fig_d1_unimodal")
fig_unimodal(D2_unimodal,"FakeTT: Unimodal Encoders","fig_d2_unimodal")

# ---------- FIG 2: Fusion strategy comparison (grouped bars) ----------
def fig_fusion(data,title,fname):
    labels=list(data.keys())
    acc=[data[k][0] for k in labels]; f1=[data[k][1] for k in labels]
    x=np.arange(len(labels)); w=0.38
    colors=[C["gray"]]*(len(labels)-1)+[C["prop"]]
    fig,ax=plt.subplots(figsize=(5.4,3.2))
    b1=ax.bar(x-w/2,acc,w,label="Accuracy",color=colors)
    b2=ax.bar(x+w/2,f1,w,label="Macro F1",color=colors,alpha=0.55)
    ax.set_xticks(x); ax.set_xticklabels(labels,rotation=15)
    ax.set_ylabel("Score (%)"); ax.set_ylim(78,90); ax.set_title(title)
    ax.legend(["Accuracy","Macro F1"],loc="lower right")
    for i,v in enumerate(acc): ax.text(i-w/2,v+0.15,f"{v:.1f}",ha="center",fontsize=7)
    save(fig,fname)

fig_fusion(D1_fusion,"Crossmodal: Fusion Strategies","fig_d1_fusion")
fig_fusion(D2_fusion,"FakeTT: Fusion Strategies","fig_d2_fusion")

# ---------- FIG 3: Ablation (horizontal bars, accuracy drop) ----------
def fig_ablation(data,title,fname):
    labels=list(data.keys()); acc=[data[k][0] for k in labels]
    full=data["Full"][0]
    colors=[C["prop"] if k=="Full" else C["b1"] for k in labels]
    fig,ax=plt.subplots(figsize=(5.0,3.0))
    y=np.arange(len(labels))
    ax.barh(y,acc,color=colors)
    ax.set_yticks(y); ax.set_yticklabels(labels); ax.invert_yaxis()
    ax.set_xlabel("Accuracy (%)"); ax.set_xlim(70,90); ax.set_title(title)
    for i,v in enumerate(acc): ax.text(v+0.2,i,f"{v:.1f}",va="center",fontsize=8)
    save(fig,fname)

fig_ablation(D1_ablation,"Crossmodal: Ablation","fig_d1_ablation")
fig_ablation(D2_ablation,"FakeTT: Ablation","fig_d2_ablation")

# ---------- FIG 4: Baseline comparison (grouped, 4 metrics) ----------
def fig_baselines(data,title,fname):
    labels=list(data.keys())
    acc=[data[k][0] for k in labels]; f1=[data[k][1] for k in labels]
    mcc=[data[k][3]*100 for k in labels]
    x=np.arange(len(labels)); w=0.27
    fig,ax=plt.subplots(figsize=(5.6,3.2))
    ax.bar(x-w,acc,w,label="Accuracy",color=C["b1"])
    ax.bar(x,f1,w,label="Macro F1",color=C["prop"])
    ax.bar(x+w,mcc,w,label="MCC×100",color=C["b3"])
    ax.set_xticks(x); ax.set_xticklabels(labels,rotation=15)
    ax.set_ylabel("Score"); ax.set_ylim(55,95); ax.set_title(title)
    ax.legend(ncol=3,loc="lower center",fontsize=8)
    save(fig,fname)

fig_baselines(D1_baselines,"Crossmodal: vs Baselines","fig_d1_baselines")
fig_baselines(D2_baselines,"FakeTT: vs Baselines","fig_d2_baselines")

# ---------- FIG 5: Cross-validation stability (per-fold + mean band) ----------
def fig_cv(cv1,cv2,fname):
    fig,ax=plt.subplots(figsize=(5.2,3.0))
    folds=np.arange(1,6)
    for cv,name,col in [(cv1,"Crossmodal",C["prop"]),(cv2,"FakeTT",C["b1"])]:
        cv=np.array(cv); ax.plot(folds,cv,"o-",color=col,label=f"{name} (μ={cv.mean():.2f})")
        ax.fill_between(folds,cv.mean()-cv.std(),cv.mean()+cv.std(),color=col,alpha=0.12)
    ax.set_xlabel("Fold"); ax.set_ylabel("Accuracy (%)"); ax.set_xticks(folds)
    ax.set_title("5-Fold Cross-Validation Stability"); ax.legend(loc="lower right")
    save(fig,fname)

fig_cv(D1_cv,D2_cv,"fig_cv_stability")

# ---------- FIG 6: Cross-dataset headline summary ----------
def fig_summary(fname):
    metrics=["Accuracy","Macro F1","ROC-AUC×100","MCC×100"]
    d1=[87.25,87.25,93.6,74.5]; d2=[85.21,84.53,91.2,69.2]
    x=np.arange(len(metrics)); w=0.38
    fig,ax=plt.subplots(figsize=(5.2,3.2))
    ax.bar(x-w/2,d1,w,label="Crossmodal",color=C["prop"])
    ax.bar(x+w/2,d2,w,label="FakeTT",color=C["b1"])
    ax.set_xticks(x); ax.set_xticklabels(metrics,rotation=10)
    ax.set_ylabel("Score"); ax.set_ylim(60,100)
    ax.set_title("Proposed Framework: Cross-Dataset Performance"); ax.legend()
    for i,v in enumerate(d1): ax.text(i-w/2,v+0.6,f"{v:.1f}",ha="center",fontsize=7)
    for i,v in enumerate(d2): ax.text(i+w/2,v+0.6,f"{v:.1f}",ha="center",fontsize=7)
    save(fig,fname)

fig_summary("fig_cross_dataset_summary")

print("\nAll figures saved to", FIGDIR)
print(os.listdir(FIGDIR))

saved: fig_d1_unimodal
saved: fig_d2_unimodal
saved: fig_d1_fusion
saved: fig_d2_fusion
saved: fig_d1_ablation
saved: fig_d2_ablation
saved: fig_d1_baselines
saved: fig_d2_baselines
saved: fig_cv_stability
saved: fig_cross_dataset_summary

All figures saved to /kaggle/working/figures
['fig_d1_fusion.pdf', 'fig_d2_unimodal.pdf', 'fig_d1_baselines.png', 'fig_d1_baselines.pdf', 'fig_d1_ablation.pdf', 'fig_d2_fusion.png', 'fig_d2_fusion.pdf', 'fig_d1_ablation.png', 'fig_cross_dataset_summary.pdf', 'fig_cv_stability.png', 'fig_d1_fusion.png', 'fig_cross_dataset_summary.png', 'fig_cv_stability.pdf', 'fig_d2_unimodal.png', 'fig_d2_ablation.png', 'fig_d1_unimodal.pdf', 'fig_d2_baselines.png', 'fig_d2_baselines.pdf', 'fig_d2_ablation.pdf', 'fig_d1_unimodal.png']


In [63]:
from sklearn.metrics import confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay

def get_preds(train, test):
    m, model = run_cfg(["vis","aud","txl","tqw"],"proposed",384,0.4,2e-3,0.01,60,64,
                       train=train,evalset=test,use_temporal=True,seed=SEED)
    model.eval()
    with torch.no_grad():
        b=move(test); logits=model(b)
        prob=torch.softmax(logits,1)[:,1].cpu().numpy()
        pred=logits.argmax(1).cpu().numpy(); true=test["lab"].numpy()
    return true,pred,prob

for tag,(tr,te,cls_names) in {
    "d1":(TR,TE,["Safe","Misleading"]),
    "d2":(FT_TR,FT_TE,["Real","Fake"])}.items():
    true,pred,prob=get_preds(tr,te)

    # confusion matrix
    cm=confusion_matrix(true,pred)
    fig,ax=plt.subplots(figsize=(3.6,3.2))
    disp=ConfusionMatrixDisplay(cm,display_labels=cls_names)
    disp.plot(ax=ax,cmap="Reds",colorbar=False,values_format="d")
    ax.set_title(f"{'Crossmodal' if tag=='d1' else 'FakeTT'}: Confusion Matrix")
    save(fig,f"fig_{tag}_confusion")

    # ROC curve
    fpr,tpr,_=roc_curve(true,prob); roc_auc=auc(fpr,tpr)
    fig,ax=plt.subplots(figsize=(4.0,3.4))
    ax.plot(fpr,tpr,color=C["prop"],lw=2,label=f"Proposed (AUC={roc_auc:.3f})")
    ax.plot([0,1],[0,1],"--",color=C["gray"],lw=1)
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(f"{'Crossmodal' if tag=='d1' else 'FakeTT'}: ROC Curve")
    ax.legend(loc="lower right")
    save(fig,f"fig_{tag}_roc")

print("Confusion + ROC saved.")

saved: fig_d1_confusion
saved: fig_d1_roc
saved: fig_d2_confusion
saved: fig_d2_roc
Confusion + ROC saved.


In [64]:
import shutil
shutil.make_archive("/kaggle/working/paper_figures","zip",FIGDIR)
print("Download: /kaggle/working/paper_figures.zip")
print("Files:", sorted(os.listdir(FIGDIR)))

Download: /kaggle/working/paper_figures.zip
Files: ['fig_cross_dataset_summary.pdf', 'fig_cross_dataset_summary.png', 'fig_cv_stability.pdf', 'fig_cv_stability.png', 'fig_d1_ablation.pdf', 'fig_d1_ablation.png', 'fig_d1_baselines.pdf', 'fig_d1_baselines.png', 'fig_d1_confusion.pdf', 'fig_d1_confusion.png', 'fig_d1_fusion.pdf', 'fig_d1_fusion.png', 'fig_d1_roc.pdf', 'fig_d1_roc.png', 'fig_d1_unimodal.pdf', 'fig_d1_unimodal.png', 'fig_d2_ablation.pdf', 'fig_d2_ablation.png', 'fig_d2_baselines.pdf', 'fig_d2_baselines.png', 'fig_d2_confusion.pdf', 'fig_d2_confusion.png', 'fig_d2_fusion.pdf', 'fig_d2_fusion.png', 'fig_d2_roc.pdf', 'fig_d2_roc.png', 'fig_d2_unimodal.pdf', 'fig_d2_unimodal.png']
